In [7]:
import sys
import types
import importlib
import numpy as np
import torch
import torch.nn.functional as F
from types import SimpleNamespace

# =========================================================
# 0. 路径配置
# =========================================================
PROJECT_DIR = "/home/mcb/users/hchen26/method/textgrad/LLM-ITL-main/scConcept_github/ECRTM"
DATA_DIR = "/home/mcb/users/hchen26/method/textgrad/LLM-ITL-main/scConcept_github/Datasets"

NEW_MODEL_PATH = "/home/mcb/users/hchen26/method/textgrad/LLM-ITL-main/scConcept_github/ECRTM/output/models/pollen_K50_seed1/epoch-500.pth"
OLD_MODEL_PATH = "/home/mcb/users/hchen26/method/textgrad/LLM-ITL-main/save_models/ECRTM_pollen_K50_seed1_useLLM-False/epoch-500.pth"

DATASET_NAME = "pollen"

# =========================================================
# 1. 导入当前工程
# =========================================================
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from models.ECRTM import ECRTM
from singlecell_dataset import SingleCellDataset

# =========================================================
# 2. 为旧 checkpoint 注册旧模块路径别名
# =========================================================
topic_models_pkg = types.ModuleType("topic_models")
topic_models_ecrtm_pkg = types.ModuleType("topic_models.ECRTM")
topic_models_models_pkg = types.ModuleType("topic_models.ECRTM.models")

sys.modules["topic_models"] = topic_models_pkg
sys.modules["topic_models.ECRTM"] = topic_models_ecrtm_pkg
sys.modules["topic_models.ECRTM.models"] = topic_models_models_pkg
sys.modules["topic_models.ECRTM.models.ECR"] = importlib.import_module("models.ECR")
sys.modules["topic_models.ECRTM.models.ECRTM"] = importlib.import_module("models.ECRTM")

# =========================================================
# 3. 构造当前新模型参数
# =========================================================
args = SimpleNamespace(
    dataset=DATASET_NAME,
    n_topic=50,
    seed=1,
    lr=0.002,
    epochs=500,
    batch_size=512,
    lr_scheduler=True,
    lr_step_size=25,
    sinkhorn_alpha=20,
    OT_max_iter=1000,
    weight_loss_ECR=100,
    eval_step=50,
    dropout=0.0,
    en1_units=200,
    beta_temp=0.2,
    data_dir=DATA_DIR,
    output_dir=f"{PROJECT_DIR}/output",
)

dataset = SingleCellDataset(
    dataset_name=args.dataset,
    batch_size=args.batch_size,
    data_dir=args.data_dir,
)

args.vocab_size = dataset.n_genes
gene_names = dataset.gene_names

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# 4. 定义兼容新旧模型的 beta 提取函数
# =========================================================
def extract_beta_compatible(model):
    """
    Extract beta from either:
    - new cleaned ECRTM model
    - old legacy ECRTM checkpoint
    """
    if not hasattr(model, "topic_embeddings"):
        raise AttributeError("Model does not have topic_embeddings.")

    topic_embeddings = model.topic_embeddings

    if hasattr(model, "gene_embeddings"):
        feature_embeddings = model.gene_embeddings
    elif hasattr(model, "word_embeddings"):
        feature_embeddings = model.word_embeddings
    else:
        raise AttributeError("Model has neither gene_embeddings nor word_embeddings.")

    if hasattr(model, "beta_temp"):
        beta_temp = model.beta_temp
    elif hasattr(model, "args") and hasattr(model.args, "beta_temp"):
        beta_temp = model.args.beta_temp
    else:
        raise AttributeError("Cannot find beta_temp in model.")

    if hasattr(model, "pairwise_euclidean_distance"):
        distance_matrix = model.pairwise_euclidean_distance(topic_embeddings, feature_embeddings)
    else:
        distance_matrix = (
            torch.sum(topic_embeddings ** 2, dim=1, keepdim=True)
            + torch.sum(feature_embeddings ** 2, dim=1)
            - 2 * torch.matmul(topic_embeddings, feature_embeddings.t())
        )

    beta = F.softmax(-distance_matrix / beta_temp, dim=0)
    return beta

# =========================================================
# 5. 加载新模型
# =========================================================
model_new = ECRTM(args).to(device)
new_ckpt = torch.load(NEW_MODEL_PATH, map_location=device)
model_new.load_state_dict(new_ckpt)

# =========================================================
# 6. 加载旧模型
# =========================================================
old_ckpt = torch.load(OLD_MODEL_PATH, map_location=device, weights_only=False)

if hasattr(old_ckpt, "get_beta") or hasattr(old_ckpt, "topic_embeddings"):
    old_model_for_eval = old_ckpt.to(device)
else:
    model_old = ECRTM(args).to(device)
    if isinstance(old_ckpt, dict):
        try:
            model_old.load_state_dict(old_ckpt)
        except Exception:
            if "state_dict" in old_ckpt:
                model_old.load_state_dict(old_ckpt["state_dict"])
            else:
                raise ValueError("Old checkpoint is not a usable model or state_dict.")
    else:
        raise ValueError("Old checkpoint format is not supported.")
    old_model_for_eval = model_old

# =========================================================
# 7. 提取 beta
# =========================================================
model_new.eval()
old_model_for_eval.eval()

with torch.no_grad():
    beta_new = extract_beta_compatible(model_new).detach().cpu().numpy()
    beta_old = extract_beta_compatible(old_model_for_eval).detach().cpu().numpy()

print("beta_new shape:", beta_new.shape)
print("beta_old shape:", beta_old.shape)

# =========================================================
# 8. 数值比较
# =========================================================
diff = np.abs(beta_new - beta_old)

print("\n===== Numeric difference of beta =====")
print("Max absolute diff :", diff.max())
print("Mean absolute diff:", diff.mean())
print("Median abs diff   :", np.median(diff))
print("Allclose atol=1e-5:", np.allclose(beta_new, beta_old, atol=1e-5))
print("Allclose atol=1e-4:", np.allclose(beta_new, beta_old, atol=1e-4))
print("Allclose atol=1e-3:", np.allclose(beta_new, beta_old, atol=1e-3))

# =========================================================
# 9. Top-gene overlap
# =========================================================
def get_top_gene_indices(beta, topk=50):
    return [np.argsort(topic_dist)[::-1][:topk] for topic_dist in beta]

def compute_topic_overlap(beta_a, beta_b, topk=50):
    top_a = get_top_gene_indices(beta_a, topk=topk)
    top_b = get_top_gene_indices(beta_b, topk=topk)

    overlaps = []
    for i in range(len(top_a)):
        set_a = set(top_a[i])
        set_b = set(top_b[i])
        overlaps.append(len(set_a & set_b) / topk)

    return np.array(overlaps), top_a, top_b

topk = 50
overlaps, top_idx_new, top_idx_old = compute_topic_overlap(beta_new, beta_old, topk=topk)

print(f"\n===== Top-{topk} gene overlap per topic =====")
for i, ov in enumerate(overlaps):
    print(f"Topic {i:02d}: overlap = {ov:.3f}")

print("\nMean overlap  :", overlaps.mean())
print("Median overlap:", np.median(overlaps))
print("Min overlap   :", overlaps.min())
print("Max overlap   :", overlaps.max())

# =========================================================
# 10. 看几个 topic 的 top genes
# =========================================================
def indices_to_gene_names(indices, gene_names):
    return [gene_names[i] for i in indices]

print("\n===== Example topic comparison =====")
for topic_id in range(min(3, beta_new.shape[0])):
    genes_new = indices_to_gene_names(top_idx_new[topic_id][:20], gene_names)
    genes_old = indices_to_gene_names(top_idx_old[topic_id][:20], gene_names)

    print(f"\nTopic {topic_id}")
    print("New top genes:")
    print(genes_new)
    print("Old top genes:")
    print(genes_old)

# =========================================================
# 11. 简单解释
# =========================================================
print("\n===== Quick interpretation =====")
if np.allclose(beta_new, beta_old, atol=1e-5):
    print("The new and old models are numerically almost identical.")
elif overlaps.mean() > 0.9:
    print("The beta matrices are not exactly identical, but the topic top genes are highly consistent.")
elif overlaps.mean() > 0.7:
    print("The overall topic structures are broadly similar, but there are noticeable differences.")
else:
    print("The new and old results differ substantially. You may need to check training logic, random seed, or data loading.")

===> Train size: 301
===> Test size: 301
===> Number of genes: 10000
===> Avg expression per cell: 7082.551
===> Number of labels: 11
beta_new shape: (50, 10000)
beta_old shape: (50, 10000)

===== Numeric difference of beta =====
Max absolute diff : 0.0
Mean absolute diff: 0.0
Median abs diff   : 0.0
Allclose atol=1e-5: True
Allclose atol=1e-4: True
Allclose atol=1e-3: True

===== Top-50 gene overlap per topic =====
Topic 00: overlap = 1.000
Topic 01: overlap = 1.000
Topic 02: overlap = 1.000
Topic 03: overlap = 1.000
Topic 04: overlap = 1.000
Topic 05: overlap = 1.000
Topic 06: overlap = 1.000
Topic 07: overlap = 1.000
Topic 08: overlap = 1.000
Topic 09: overlap = 1.000
Topic 10: overlap = 1.000
Topic 11: overlap = 1.000
Topic 12: overlap = 1.000
Topic 13: overlap = 1.000
Topic 14: overlap = 1.000
Topic 15: overlap = 1.000
Topic 16: overlap = 1.000
Topic 17: overlap = 1.000
Topic 18: overlap = 1.000
Topic 19: overlap = 1.000
Topic 20: overlap = 1.000
Topic 21: overlap = 1.000
Topic 22